In [2]:
import pandas as pd
# /page = pd.read_html("https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Egypt")
import requests

CITY_Coordinates = {
    "Cairo": {"lat": 30.0444, "lon": 31.2357},
    "Alexandria": {"lat": 31.2001, "lon": 29.9187},
    "Giza": {"lat": 30.0131, "lon": 31.2089},
    "Luxor": {"lat": 25.6872, "lon": 32.6396},
    "Aswan": {"lat": 24.0889, "lon": 32.8998},
    "Port Said": {"lat": 31.2653, "lon": 32.3019},
    "Suez": {"lat": 29.9668, "lon": 32.5498},
    "Hurghada": {"lat": 27.2579, "lon": 33.8116},
    "Sharm El Sheikh": {"lat": 27.9158, "lon": 34.3300},
    "Mansoura": {"lat": 31.0409, "lon": 31.3785},
}

In [3]:
lat=CITY_Coordinates["Cairo"]["lat"]
lon=CITY_Coordinates["Cairo"]["lon"]
response =requests.get('https://api.open-meteo.com/v1/forecast',params={
    "latitude":lat,
    "longitude": lon,
    "current_weather":True,
})
data= response.json()
data

{'latitude': 30.0625,
 'longitude': 31.25,
 'generationtime_ms': 0.06830692291259766,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 22.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-07-22T15:45',
  'interval': 900,
  'temperature': 38.3,
  'windspeed': 16.9,
  'winddirection': 322,
  'is_day': 1,
  'weathercode': 1}}

In [4]:
current =data["current_weather"]
current_weather_units =data["current_weather_units"]
print(f"temperature => {current['temperature']}{current_weather_units['temperature']}")


temperature => 38.3°C


In [ ]:
import time

Weather_records=[]
# print(CITY_Coordinates.items())
# print("#"*25)
# print(CITY_Coordinates.values())
for city in CITY_Coordinates.items():
    response =requests.get('https://api.open-meteo.com/v1/forecast',params={
      "latitude":city[1]["lat"],
      "longitude": city[1]["lon"],
      "current_weather":True,
    })
    if response.status_code==200:
      data= response.json()
      current =data["current_weather"]
      Weather_records.append({
          "city":city[0],
          "temperature":current["temperature"],
          "windspeed":current["windspeed"],
          "time":current["time"],
      })
    time.sleep(0.3)
Weather_records

In [ ]:
Weather_df=pd.DataFrame(Weather_records)
Weather_df

,city,temperature,windspeed,time
0,Cairo,40.1,13.7,2026-07-22T14:45
1,Alexandria,29.4,16.1,2026-07-22T14:45
2,Giza,39.9,15.8,2026-07-22T14:45
3,Luxor,41.4,16.8,2026-07-22T14:45
4,Aswan,41.5,18.9,2026-07-22T14:45
5,Port Said,31.9,22.6,2026-07-22T14:45
6,Suez,40.7,16.1,2026-07-22T14:45
7,Hurghada,34.7,23.8,2026-07-22T14:45
8,Sharm El Sheikh,40.5,5.4,2026-07-22T14:45
9,Mansoura,36.1,17.6,2026-07-22T14:45


In [ ]:
HEADERS = {"User-Agent":"DigitalEgyptCubEduBot/1.0 (edu)"}
page=requests.get("https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Egypt",headers=HEADERS)
print(page.status_code)

200


In [ ]:
# BeautifulSoup
from bs4 import BeautifulSoup
soup=BeautifulSoup(page.text,"html.parser")
print("Title", soup.find("h1").text )
tabels=soup.find_all("table",class_="wikitable")
firsttableheader=[th.text.strip() for th in tabels[0].find_all("th")]
print(firsttableheader)

Title List of cities and towns in Egypt
['Name', 'Arabic', 'Governorate', 'Area code', 'Population (2023 estimate)[11]', 'Photo']


In [ ]:
# StringIO
from io import StringIO
all_tables=pd.read_html(StringIO(page.text))
print("number of tabels pandas found:",len(all_tables))

city_table=None
for t in all_tables:
    col_names=[str(c).lower() for c in t.columns]
    if any("population" in c for c in col_names) and  any("city" in c or "name" in c for c in col_names) :
      city_table=t
      break
city_table

number of tabels pandas found: 15


,Name,Arabic,Governorate,Area code,Population (2023 estimate)[11],Photo
0,Cairo*,القاهرة,Cairo,(+20) 2,10000000,NaN
1,Alexandria,الاسكندرية,Alexandria,(+20) 3,5362517,NaN
2,Giza*,الجيزة,Giza,(+20) 2,4458135,NaN
3,Shubra El Kheima*,شبرا الخيمة,Qalyubia,(+20) 2,1275700,NaN
4,Port Said,بور سعيد,Port Said,(+20) 60,791749,NaN
5,Suez,السويس,Suez,(+20) 60,716458,NaN
6,Mansoura,المنصورة,Dakahlia,(+20) 50,632330,NaN
7,El Mahalla El Kubra,المحلة الكبرى,Gharbia,(+20) 40,614202,NaN
8,Tanta,طنطا,Gharbia,(+20) 40,597694,NaN
9,Asyut,اسيوط,Asyut,(+20) 88,562061,NaN


In [ ]:
CITY_COL="Name"
GOV_COL=[ c for c in city_table.columns if "gov" in str(c).lower()][0]
POP_COL=[ c for c in city_table.columns if "popu" in str(c).lower()][0]
print(GOV_COL,POP_COL)
target_city=list(CITY_Coordinates.keys())
print(target_city)
city_table["name_Clean"]=city_table[CITY_COL].str.replace("*","").str.strip().str.title();

station_df=city_table[city_table["name_Clean"].isin(target_city)][["name_Clean", GOV_COL , POP_COL]].copy()
station_df.columns=[ "city","governorate","population"]
station_df=station_df.reset_index(drop=True)
station_df


Governorate Population (2023 estimate)[11]
['Cairo', 'Alexandria', 'Giza', 'Luxor', 'Aswan', 'Port Said', 'Suez', 'Hurghada', 'Sharm El Sheikh', 'Mansoura']


,city,governorate,population
0,Cairo,Cairo,10000000
1,Alexandria,Alexandria,5362517
2,Giza,Giza,4458135
3,Port Said,Port Said,791749
4,Suez,Suez,716458
5,Mansoura,Dakahlia,632330
6,Aswan,Aswan,401890
7,Luxor,Luxor,284952
8,Hurghada,Red Sea,214247


In [ ]:
# Combined
Combined_df=pd.merge(Weather_df,station_df,on="city")
Combined_df

,city,temperature,windspeed,time,governorate,population
0,Cairo,40.1,13.7,2026-07-22T14:45,Cairo,10000000
1,Alexandria,29.4,16.1,2026-07-22T14:45,Alexandria,5362517
2,Giza,39.9,15.8,2026-07-22T14:45,Giza,4458135
3,Luxor,41.4,16.8,2026-07-22T14:45,Luxor,284952
4,Aswan,41.5,18.9,2026-07-22T14:45,Aswan,401890
5,Port Said,31.9,22.6,2026-07-22T14:45,Port Said,791749
6,Suez,40.7,16.1,2026-07-22T14:45,Suez,716458
7,Hurghada,34.7,23.8,2026-07-22T14:45,Red Sea,214247
8,Mansoura,36.1,17.6,2026-07-22T14:45,Dakahlia,632330


In [ ]:
# export data
Combined_df.to_csv("eg_weather.csv",index=False)

In [ ]:
city_weather=pd.read_csv("eg_weather.csv")
city_weather.info()
city_weather=city_weather.dropna()
print(city_weather.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   city         9 non-null      object 
 1   temperature  9 non-null      float64
 2   windspeed    9 non-null      float64
 3   time         9 non-null      object 
 4   governorate  9 non-null      object 
 5   population   9 non-null      int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 564.0+ bytes
city           0
temperature    0
windspeed      0
time           0
governorate    0
population     0
dtype: int64


In [ ]:
city_weather.head()

,city,temperature,windspeed,time,governorate,population
0,Cairo,40.1,13.7,2026-07-22T14:45,Cairo,10000000
1,Alexandria,29.4,16.1,2026-07-22T14:45,Alexandria,5362517
2,Giza,39.9,15.8,2026-07-22T14:45,Giza,4458135
3,Luxor,41.4,16.8,2026-07-22T14:45,Luxor,284952
4,Aswan,41.5,18.9,2026-07-22T14:45,Aswan,401890


In [ ]:
city_weather.describe()

,temperature,windspeed,population
count,9.000000,9.000000,9.000000e+00
mean,37.300000,17.933333,2.540253e+06
std,4.477443,3.311344,3.398239e+06
min,29.400000,13.700000,2.142470e+05
25%,34.700000,16.100000,4.018900e+05
50%,39.900000,16.800000,7.164580e+05
75%,40.700000,18.900000,4.458135e+06
max,41.500000,23.800000,1.000000e+07


In [ ]:
city_weather.shape

(9, 6)

In [ ]:
city_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   city         9 non-null      object 
 1   temperature  9 non-null      float64
 2   windspeed    9 non-null      float64
 3   time         9 non-null      object 
 4   governorate  9 non-null      object 
 5   population   9 non-null      int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 564.0+ bytes


In [ ]:
print(f"null values in the dataset => {city_weather.isna().sum().sum()}")


null values in the dataset => 0


In [ ]:
city_weather.describe()

,temperature,windspeed,population
count,9.000000,9.000000,9.000000e+00
mean,37.300000,17.933333,2.540253e+06
std,4.477443,3.311344,3.398239e+06
min,29.400000,13.700000,2.142470e+05
25%,34.700000,16.100000,4.018900e+05
50%,39.900000,16.800000,7.164580e+05
75%,40.700000,18.900000,4.458135e+06
max,41.500000,23.800000,1.000000e+07


In [ ]:
print(f"For TESTING " )

if city_weather.duplicated().sum()>0:
    print(f"Duplicate values in the dataset => {city_weather.duplicated().sum()}")
else:
    print("No duplicate values in the dataset.")

For TESTING 
No duplicate values in the dataset.


In [ ]:
print(f" Max temperature => {city_weather['temperature'].max()} in city {city_weather.loc[city_weather['temperature'].idxmax(),'city']}")

 Max temperature => 41.5 in city Aswan


In [ ]:
print(f" Max temperature => {city_weather['temperature'].max()} in city {city_weather.loc[city_weather['temperature'].idxmax(),'city']}")

 Max temperature => 41.5 in city Aswan


In [ ]:
city_weather.describe()

,temperature,windspeed,population
count,9.000000,9.000000,9.000000e+00
mean,37.300000,17.933333,2.540253e+06
std,4.477443,3.311344,3.398239e+06
min,29.400000,13.700000,2.142470e+05
25%,34.700000,16.100000,4.018900e+05
50%,39.900000,16.800000,7.164580e+05
75%,40.700000,18.900000,4.458135e+06
max,41.500000,23.800000,1.000000e+07


In [ ]:
print(f"For TESTING " )

if city_weather.duplicated().sum()>0:
    print(f"Duplicate values in the dataset => {city_weather.duplicated().sum()}")
else:
    print("No duplicate values in the dataset.")

For TESTING 
No duplicate values in the dataset.


In [ ]:
# bilal 
city_weather.info()

NameError: name 'city_weather' is not defined